In [ ]:
import yfinance as yf
import pywt
import numpy as np
import pandas as pd
import scipy.linalg as linalg
from scipy.optimize import minimize
from scipy.stats import chisquare
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from collections import OrderedDict
from sklearn.covariance import LedoitWolf

## Wavelet reconstruction

In [ ]:
def reconstruct_levels(signal, wavelet='db4', level=4):
    coeffs = pywt.wavedec(signal, wavelet, level=level)
    reconstructed = {}

    # --- Reconstruct approximation ---
    coeffs_A = [coeffs[0]] + [np.zeros_like(c) for c in coeffs[1:]]
    A = pywt.waverec(coeffs_A, wavelet)
    reconstructed[f"A{level}"] = A[:len(signal)]

    # --- Reconstruct each detail ---
    for j in range(1, level + 1):
        coeffs_D = [np.zeros_like(c) for c in coeffs]
        coeffs_D[j] = coeffs[j]
        D = pywt.waverec(coeffs_D, wavelet)
        reconstructed[f"D{level - j + 1}"] = D[:len(signal)]

    return reconstructed

## ARIMA-GARCH model

In [ ]:
def ARIMAGARCH(data):
  def arma_garch_loglik(params, y):
      mu, phi, theta, omega, alpha, beta = params

      T = len(y)
      eps = np.zeros(T)
      sigma2 = np.zeros(T)

      # Initialize
      sigma2[0] = np.var(y)

      for t in range(1, T):
          # Mean equation
          eps[t] = y[t] - mu - phi*y[t-1] - theta*eps[t-1]

          # Variance equation
          sigma2[t] = omega + alpha*eps[t-1]**2 + beta*sigma2[t-1]

          # Prevent negative variance
          if sigma2[t] <= 0:
              return 1e10

      loglik = -0.5 * np.sum(
          np.log(2*np.pi) +
          np.log(sigma2) +
          eps**2 / sigma2
      )

      return -loglik  # minimize negative log-likelihood

  # Initial guesses
  init_params = np.array([
      np.mean(data),   # mu
      0.1,          # phi
      0.1,          # theta
      0.1*np.var(data),# omega
      0.1,          # alpha
      0.8           # beta
  ])

  bounds = [
      (None, None),   # mu
      (-0.99, 0.99),  # phi
      (-0.99, 0.99),  # theta
      (1e-6, None),   # omega
      (1e-6, 1),      # alpha
      (1e-6, 1)       # beta
  ]

  result = minimize(
      arma_garch_loglik,
      init_params,
      args=(data,),
      bounds=bounds,
      method="L-BFGS-B"
  )

  return result

## Wavelet-ARIMA-GARCH

In [ ]:
def compute_variance(params, data):
  mu, omega, alpha, beta = params
  T = len(data)

  eps = data - mu
  sigma2 = np.zeros(T)

  # Initialize with unconditional variance
  sigma2[0] = np.var(data)

  for t in range(1, T):
      sigma2[t] = omega + alpha * eps[t-1]**2 + beta * sigma2[t-1]

  return sigma2

In [ ]:
def get_metrics(actual, pred, var):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)

    # Hit Ratio (Percentage of points inside 95% bands)
    upper = pred + 1.96 * np.sqrt(var)
    lower = pred - 1.96 * np.sqrt(var)
    hit_ratio = np.mean((actual <= upper) & (actual >= lower)) * 100

    return {"RMSE": rmse, "MAE": mae, "HitRatio": hit_ratio}

# Compare them
# wavelet_metrics = get_metrics(df['log_return'], total_mean_forecast, total_var)
# normal_metrics = get_metrics(df['log_return'], mean_normal, variance_normal)

# print(f"Wavelet Model: {wavelet_metrics}")
# print(f"Normal Model:  {normal_metrics}")

# rmse_zero = np.sqrt(np.mean(df['log_return']**2))
# print(rmse_zero)

In [ ]:
def chi_squared_test(actual, pred):
  stat, p = chisquare(f_obs=actual[-100:], f_exp=pred[-100:])
  print(f"P-value: {p}")


## Combined Wavelet ARIMA GARCH Code

In [ ]:
def wARIMAGARCH(df, wavelet='db4', level=4, mode='full'):
  # Decompose and reconstruct levels
  reconstructed = reconstruct_levels(df['log_return'].values, wavelet, level)

  # Dictionary to hold ARIMA-GARCH parameters for each level
  model_results = {}

  # Iterate ARIMA-GARCH for each level
  for name, signal in reconstructed.items():
      print(f"Optimizing {name}...")
      try:
          result = ARIMAGARCH(signal)

          if result.success:
              model_results[name] = result.x
              print(f"Successfully optimized {name}")
          else:
              print(f"Optimization failed for {name}: {result.message}")

      except Exception as e:
          print(f"Error processing {name}: {e}")

  # Return variables
  total_mean_forecast = np.zeros(len(df))
  total_upper_band = np.zeros(len(df))
  total_lower_band = np.zeros(len(df))
  total_var = np.zeros(len(df))

  # Iterate ARIMA-GARCH forecast for each level
  for name, signal in reconstructed.items():
      mu, phi, theta, omega, alpha, beta = model_results[name]

      var = compute_variance((mu, omega, alpha, beta), signal)

      level_mean = np.zeros(len(signal))
      eps = np.zeros(len(signal))

      for i in range(1, len(signal)):
          eps[i-1] = signal[i-1] - level_mean[i-1]

          level_mean[i] = mu + phi * signal[i-1] + theta * eps[i-1]

          total_mean_forecast[i] += level_mean[i]

          total_var[i] += var[i]

  total_upper_band = total_mean_forecast + 1.96 * np.sqrt(total_var)
  total_lower_band = total_mean_forecast - 1.96 * np.sqrt(total_var)

  if mode == 'full':
    return total_mean_forecast, total_upper_band, total_lower_band
  else:
    return total_mean_forecast


In [ ]:
def ARIMAGARCH_forecast(df):
  # Fit normal model on raw returns
  res_normal = ARIMAGARCH(df['log_return'].values)
  mu_n, phi_n, theta_n, omega_n, alpha_n, beta_n = res_normal.x

  # Generate the Normal forecast and variance
  # (Assuming you have a 'variance_normal' from compute_variance)
  variance_normal = compute_variance((mu_n, omega_n, alpha_n, beta_n), df['log_return'].values)
  mean_normal = np.zeros(len(df))
  resid_normal = np.zeros(len(df))

  for i in range(1, len(df)):
      resid_normal[i-1] = df['log_return'].values[i-1] - mean_normal[i-1]
      mean_normal[i] = mu_n + phi_n * df['log_return'].values[i-1] + theta_n * resid_normal[i-1]

  return mean_normal, variance_normal

In [ ]:
def zoomed_plot(df, total_mean_forecast, total_upper_band, total_lower_band, mean_normal, variance_normal, ticker, zoom_window=100):
  # Extract the zoom window (e.g., last 100 points)
  zoom_df = df.iloc[-1*zoom_window:]

  plt.figure(figsize=(15, 7))

  # Actual Returns (Zoomed)
  plt.plot(zoom_df.index, zoom_df['log_return'], label=f'Actual {ticker} Returns',
          alpha=0.4, color='black', marker='o', markersize=3, zorder=1)

  # Normal ARIMA-GARCH (Red)
  # Slicing the pre-computed arrays to match the zoom window
  plt.plot(zoom_df.index, mean_normal[-zoom_window:],
          label='Normal ARIMA-GARCH Mean', color='red', lw=2, zorder=2)
  plt.fill_between(zoom_df.index,
                  (mean_normal - 1.96*np.sqrt(variance_normal))[-zoom_window:],
                  (mean_normal + 1.96*np.sqrt(variance_normal))[-zoom_window:],
                  color='red', alpha=0.1)

  # Wavelet-ARIMA-GARCH (Blue/Orange)
  plt.plot(zoom_df.index, total_mean_forecast[-zoom_window:],
          label='Wavelet-ARIMA-GARCH Mean', color='blue', lw=2, zorder=3)
  plt.fill_between(zoom_df.index,
                  total_lower_band[-zoom_window:],
                  total_upper_band[-zoom_window:],
                  color='orange', alpha=0.2, label='Wavelet 95% Envelope')

  # Formatting for clarity
  plt.title(f"Zoomed Comparison (Last {zoom_window} Days): Normal vs. Wavelet ARIMA-GARCH", fontsize=14)
  plt.xlabel("Date")
  plt.ylabel("Log Returns")
  plt.legend(loc='upper left', frameon=True)
  plt.grid(True, alpha=0.3)
  plt.xticks(rotation=45) # Rotate dates for better readability
  plt.tight_layout()


## Mean-Variance Portfolio

In [ ]:
#================================
# data download and clean-up
#================================
def download_prices_and_returns(
    tickers,
    period="5y",
    interval="1d",
    auto_adjust=False,
):
    data = yf.download(
        tickers=tickers,
        period=period,
        interval=interval,
        auto_adjust=auto_adjust,
        progress=False,
        group_by="column",
    )

    if data.empty:
        raise ValueError("No data was downloaded.")

    if isinstance(data.columns, pd.MultiIndex):
        if "Adj Close" in data.columns.get_level_values(0):
            price_df = data["Adj Close"].copy()
        elif "Close" in data.columns.get_level_values(0):
            price_df = data["Close"].copy()
        else:
            raise ValueError("Neither 'Adj Close' nor 'Close' found.")
    else:
        if "Adj Close" in data.columns:
            price_df = data[["Adj Close"]].copy()
            price_df.columns = tickers[:1]
        elif "Close" in data.columns:
            price_df = data[["Close"]].copy()
            price_df.columns = tickers[:1]
        else:
            raise ValueError("Neither 'Adj Close' nor 'Close' found.")

    if isinstance(price_df, pd.Series):
        price_df = price_df.to_frame(name=tickers[0])

    price_df = price_df.dropna(how="any")
    log_return_df = np.log(price_df / price_df.shift(1)).dropna(how="any")
    return price_df, log_return_df

#===============================
# Mean-variance portfolio
#===============================

def mean_variance_portfolio_optimization(mu_vec, cov_matrix, risk_aversion=1.0, eps=1e-8):
    """
    Forecast-aware long-only optimizer:

        maximize  mu'w - 0.5 * lambda * w'Σw
        subject to sum(w)=1, w>=0
    """
    if not isinstance(mu_vec, pd.Series):
        mu_vec = pd.Series(mu_vec, index=cov_matrix.columns)

    assets = list(cov_matrix.columns)
    mu = mu_vec.loc[assets].to_numpy(dtype=float)
    Sigma = cov_matrix.loc[assets, assets].to_numpy(dtype=float) + eps * np.eye(len(assets)) #small eps to ensure invertibility

    n = len(assets)
    x0 = np.repeat(1.0 / n, n)
    bounds = [(0.0, 1.0) for _ in range(n)]
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

    def objective(w):
        return -(mu @ w - 0.5 * risk_aversion * (w @ Sigma @ w))

    result = minimize(
        objective,
        x0=x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 1000, "ftol": 1e-12},
    )

    if not result.success:
        raise RuntimeError(f"Forecast-aware optimization failed: {result.message}")

    w = result.x / result.x.sum()
    return pd.Series(w, index=assets, name="weight")

def forecast_mu_and_cov(
    window_returns,
    use_mean_forecast=True,
    rebalance_frequency="weekly",
    use_shrinkage=True,
):
    """
    Build forecasted mean vector and covariance matrix.

    Parameters
    ----------
    window_returns : pd.DataFrame
        Trailing daily log returns.
    use_mean_forecast : bool
        If False, sets expected returns to zero and uses only risk forecast.
    rebalance_frequency : str
        'daily', 'weekly', or 'monthly'
    use_shrinkage : bool
        If True, use shrinkage covariance for correlation estimation.

    Returns
    -------
    mu_hat : pd.Series
        Expected return vector over the holding period.
    Sigma_hat : pd.DataFrame
        Forecast covariance matrix over the holding period.
    fit_info : dict
        Per-asset forecast details.
    """
    assets = list(window_returns.columns)
    n = len(assets)

    if rebalance_frequency == "daily":
        horizon = 1
    elif rebalance_frequency == "weekly":
        horizon = 5
    elif rebalance_frequency == "monthly":
        horizon = 21
    else:
        raise ValueError("rebalance_frequency must be 'daily', 'weekly', or 'monthly'")

    if use_shrinkage:
        base_cov = estimate_shrinkage_covariance(window_returns)
    else:
        base_cov = window_returns.cov()

    base_corr = covariance_to_correlation(base_cov)

    mu_hat = np.zeros(n)
    var_hat = np.zeros(n)
    fit_info = {}

    for i, asset in enumerate(assets):
        fc = multi_step_arima_garch_forecast(
            window_returns[asset].values,
            horizon=horizon,
        )

        if use_mean_forecast:
            if horizon == 1:
                mu_hat[i] = fc["mean_forecasts"][0]
            else:
                mu_hat[i] = fc["mean_forecasts"].sum()
        else:
            mu_hat[i] = 0.0

        if horizon == 1:
            var_hat[i] = fc["var_forecasts"][0]
        else:
            var_hat[i] = fc["var_forecasts"].sum()

        fit_info[asset] = fc

    D = np.diag(np.sqrt(np.clip(var_hat, 1e-12, None)))
    Sigma_hat = D @ base_corr.values @ D

    mu_hat = pd.Series(mu_hat, index=assets, name="mu_forecast")
    Sigma_hat = pd.DataFrame(Sigma_hat, index=assets, columns=assets)

    return mu_hat, Sigma_hat, fit_info

def multi_step_arima_garch_forecast(signal_window, horizon=5):
    """
    Fit ARIMA-GARCH on a trailing window and produce multi-step forecasts.

    Parameters
    ----------
    signal_window : array-like
        Trailing return window for one asset and one wavelet level.
    horizon : int
        Forecast horizon, e.g. 5 for weekly if using daily data.

    Returns
    -------
    dict
        {
            "params": fitted params,
            "mean_forecasts": array of length horizon,
            "var_forecasts": array of length horizon,
            "weekly_mean": aggregated mean forecast,
            "weekly_var": aggregated variance forecast
        }
    """
    y = np.asarray(signal_window, dtype=float)
    result = ARIMAGARCH(y)

    if not result.success:
        raise RuntimeError(f"ARIMAGARCH optimization failed: {result.message}")

    mu, phi, theta, omega, alpha, beta = result.x
    T = len(y)

    # Reconstruct in-sample residuals and conditional variances
    eps = np.zeros(T)
    sigma2 = np.zeros(T)
    sigma2[0] = np.var(y)

    for t in range(1, T):
        eps[t] = y[t] - mu - phi * y[t - 1] - theta * eps[t - 1]
        sigma2[t] = omega + alpha * eps[t - 1] ** 2 + beta * sigma2[t - 1]
        sigma2[t] = max(sigma2[t], 1e-10)

    last_y = y[-1]
    last_eps = eps[-1]
    last_sigma2 = sigma2[-1]

    mean_forecasts = np.zeros(horizon)
    var_forecasts = np.zeros(horizon)

    # Step 1
    mean_forecasts[0] = mu + phi * last_y + theta * last_eps
    var_forecasts[0] = omega + alpha * last_eps**2 + beta * last_sigma2
    var_forecasts[0] = max(var_forecasts[0], 1e-10)

    # Step 2+
    # Future shocks have expectation zero, so MA effect drops out after step 1
    for h in range(1, horizon):
        mean_forecasts[h] = mu + phi * mean_forecasts[h - 1]
        var_forecasts[h] = omega + (alpha + beta) * var_forecasts[h - 1]
        var_forecasts[h] = max(var_forecasts[h], 1e-10)

    weekly_mean = mean_forecasts.sum()

    # Approximation: sum conditional variances across days
    # Ignores cross-horizon covariance terms
    weekly_var = var_forecasts.sum()

    return {
        "params": result.x,
        "mean_forecasts": mean_forecasts,
        "var_forecasts": var_forecasts,
        "weekly_mean": float(weekly_mean),
        "weekly_var": float(weekly_var),
    }

#=================================================
# Shrinkage estimator for the cov matrix
#=================================================

def estimate_shrinkage_covariance(window_returns):
    """
    Estimate covariance using Ledoit-Wolf shrinkage.
    """
    lw = LedoitWolf()
    lw.fit(window_returns.values)

    cov_df = pd.DataFrame(
        lw.covariance_,
        index=window_returns.columns,
        columns=window_returns.columns,
    )
    return cov_df


def covariance_to_correlation(cov_df, eps=1e-12):
    """
    Convert covariance matrix to correlation matrix.
    """
    std = np.sqrt(np.clip(np.diag(cov_df.values), eps, None))
    corr = cov_df.values / np.outer(std, std)
    corr = np.clip(corr, -1.0, 1.0)
    np.fill_diagonal(corr, 1.0)

    return pd.DataFrame(corr, index=cov_df.index, columns=cov_df.columns)

## Backtest layer

In [ ]:
def compute_rebalance_dates(index, frequency="weekly"):
    """
    Generate rebalance dates from a DatetimeIndex.
    """
    if frequency == "daily":
        return index

    index_series = pd.Series(index=index, data=index)

    if frequency == "weekly":
        return pd.DatetimeIndex(index_series.groupby(index.to_period("W")).last().values)

    if frequency == "monthly":
        return pd.DatetimeIndex(index_series.groupby(index.to_period("M")).last().values)

    raise ValueError("frequency must be one of {'daily', 'weekly', 'monthly'}")

def aggregate_forward_returns(return_df, start_loc, horizon):
    """
    Aggregate future daily log returns over the holding period.

    Parameters
    ----------
    return_df : pd.DataFrame
        Daily log return DataFrame.
    start_loc : int
        Integer location of rebalance date.
    horizon : int
        Number of trading days to hold.

    Returns
    -------
    forward_returns : pd.Series or None
        Aggregated future log returns by asset.
    end_date : pd.Timestamp or None
        End date of the holding period.
    """
    end_loc = start_loc + horizon

    if end_loc >= len(return_df):
        return None, None

    future_slice = return_df.iloc[start_loc + 1 : end_loc + 1]
    forward_returns = future_slice.sum(axis=0)
    end_date = return_df.index[end_loc]

    return forward_returns, end_date

#===============================================
# Backtesting portfolio
#===============================================

def rolling_mean_variance_backtest(
    log_return_df,
    window_size=252,
    rebalance_frequency="weekly",
    use_mean_forecast=True,
    use_shrinkage=True,
    risk_aversion=1.0,
):
    """
    Rolling mean-variance portfolio backtest.

    Parameters
    ----------
    log_return_df : pd.DataFrame
        Daily asset log returns.
    window_size : int
        Lookback window for fitting forecasts and covariance.
    rebalance_frequency : str
        Default is 'weekly'.
    use_mean_forecast : bool
        If False, this becomes a risk-only portfolio.
    use_shrinkage : bool
        If True, use Ledoit-Wolf shrinkage.
    risk_aversion : float
        Risk-aversion parameter in mean-variance optimization.

    Returns
    -------
    weights_df : pd.DataFrame
        Portfolio weights at rebalance dates.
    portfolio_log_returns : pd.Series
        Holding-period portfolio log returns.
    mu_forecasts_df : pd.DataFrame
        Forecast means at rebalance dates.
    summary : pd.Series
        Portfolio performance summary.
    """
    log_return_df = log_return_df.dropna(how="any").copy()

    if rebalance_frequency == "daily":
        horizon = 1
    elif rebalance_frequency == "weekly":
        horizon = 5
    elif rebalance_frequency == "monthly":
        horizon = 21
    else:
        raise ValueError("rebalance_frequency must be 'daily', 'weekly', or 'monthly'")

    rebalance_dates = compute_rebalance_dates(log_return_df.index, rebalance_frequency)

    weights_list = []
    forecast_list = []
    realized_list = []

    total_steps = len(rebalance_dates)

    for i, date in enumerate(rebalance_dates):

        if i % 10 == 0 or i == total_steps - 1:
            progress = (i + 1) / total_steps * 100
            print(f"Progress: {progress:.1f}% ({i+1}/{total_steps})")

        end_loc = log_return_df.index.get_loc(date)

        if isinstance(end_loc, slice):
            end_loc = end_loc.stop - 1

        if end_loc < window_size - 1:
            continue

        window_returns = log_return_df.iloc[end_loc - window_size + 1 : end_loc + 1]

        if len(window_returns) < window_size:
            continue

        try:
            mu_hat, Sigma_hat, fit_info = forecast_mu_and_cov(
                window_returns=window_returns,
                use_mean_forecast=use_mean_forecast,
                rebalance_frequency=rebalance_frequency, # daily for rebalance based on tomorrow's return
                use_shrinkage=use_shrinkage,
            )

            w_t = mean_variance_portfolio_optimization(
                mu_vec=mu_hat,
                cov_matrix=Sigma_hat,
                risk_aversion=risk_aversion,
            )

            forward_returns, realized_date = aggregate_forward_returns(
                log_return_df,
                start_loc=end_loc,
                horizon=horizon,
            )

            if forward_returns is None:
                continue

            realized_portfolio_log_return = float(forward_returns.loc[w_t.index] @ w_t.values)

            w_t.name = date
            weights_list.append(w_t)

            mu_row = mu_hat.copy()
            mu_row.name = date
            forecast_list.append(mu_row)

            realized_list.append(
                pd.Series(
                    {
                        "rebalance_date": date,
                        "realized_date": realized_date,
                        "portfolio_log_return": realized_portfolio_log_return,
                    }
                )
            )

        except Exception as e:
            print(f"Skipping {date} due to error: {e}")

    if not weights_list:
        raise ValueError("No valid portfolio weights were computed.")

    weights_df = pd.DataFrame(weights_list)
    weights_df.index.name = "Date"

    mu_forecasts_df = pd.DataFrame(forecast_list)
    mu_forecasts_df.index.name = "Date"

    realized_df = pd.DataFrame(realized_list)
    portfolio_log_returns = pd.Series(
        realized_df["portfolio_log_return"].values,
        index=pd.DatetimeIndex(realized_df["realized_date"]),
        name="portfolio_log_return",
    )

    summary, cumulative_growth, simple_returns = summarize_portfolio_from_log_returns(
        portfolio_log_returns,
        rebalance_frequency=rebalance_frequency,
    )

    return weights_df, portfolio_log_returns, mu_forecasts_df, summary, cumulative_growth, simple_returns

#=================================================
# Summary of performance
#=================================================
def summarize_portfolio_from_log_returns(portfolio_log_returns, rebalance_frequency="weekly"):
    """
    Summarize portfolio performance using simple returns.
    """
    simple_returns = np.exp(portfolio_log_returns) - 1.0
    simple_returns.name = "portfolio_simple_return"

    cumulative_growth = (1.0 + simple_returns).cumprod()
    cumulative_growth.name = "cumulative_growth"

    n_periods = len(simple_returns)

    if rebalance_frequency == "daily":
        periods_per_year = 252
    elif rebalance_frequency == "weekly":
        periods_per_year = 52
    elif rebalance_frequency == "monthly":
        periods_per_year = 12
    else:
        raise ValueError("rebalance_frequency must be 'daily', 'weekly', or 'monthly'")

    total_return = cumulative_growth.iloc[-1] - 1.0 if n_periods > 0 else np.nan
    annualized_return = (
        cumulative_growth.iloc[-1] ** (periods_per_year / n_periods) - 1.0
        if n_periods > 0 else np.nan
    )
    annualized_volatility = simple_returns.std() * np.sqrt(periods_per_year)
    sharpe_like = (
        simple_returns.mean() / simple_returns.std() * np.sqrt(periods_per_year)
        if simple_returns.std() > 0 else np.nan
    )

    summary = pd.Series(
        {
            "mean_period_simple_return": simple_returns.mean(),
            "total_simple_return": total_return,
            "annualized_simple_return": annualized_return,
            "annualized_volatility": annualized_volatility,
            "sharpe_like": sharpe_like,
            "final_cumulative_growth": cumulative_growth.iloc[-1] if n_periods > 0 else np.nan,
        }
    )

    return summary, cumulative_growth, simple_returns


## Buy and hold model

In [ ]:
def equal_weight_benchmark(
    log_return_df,
    rebalance_frequency="weekly",
):
    """
    Equal-weight benchmark using the same holding-period aggregation.
    """
    log_return_df = log_return_df.dropna(how="any").copy()

    if rebalance_frequency == "daily":
        horizon = 1
    elif rebalance_frequency == "weekly":
        horizon = 5
    elif rebalance_frequency == "monthly":
        horizon = 21
    else:
        raise ValueError("rebalance_frequency must be 'daily', 'weekly', or 'monthly'")

    rebalance_dates = compute_rebalance_dates(log_return_df.index, rebalance_frequency)
    n_assets = log_return_df.shape[1]
    w_eq = np.repeat(1.0 / n_assets, n_assets)

    realized_list = []

    for date in rebalance_dates:
        end_loc = log_return_df.index.get_loc(date)

        if isinstance(end_loc, slice):
            end_loc = end_loc.stop - 1

        forward_returns, realized_date = aggregate_forward_returns(
            log_return_df,
            start_loc=end_loc,
            horizon=horizon,
        )

        if forward_returns is None:
            continue

        realized_portfolio_log_return = float(forward_returns.values @ w_eq)

        realized_list.append(
            pd.Series(
                {
                    "rebalance_date": date,
                    "realized_date": realized_date,
                    "portfolio_log_return": realized_portfolio_log_return,
                }
            )
        )

    realized_df = pd.DataFrame(realized_list)

    portfolio_log_returns = pd.Series(
        realized_df["portfolio_log_return"].values,
        index=pd.DatetimeIndex(realized_df["realized_date"]),
        name="equal_weight_log_return",
    )

    summary, cumulative_growth, simple_returns = summarize_portfolio_from_log_returns(
        portfolio_log_returns,
        rebalance_frequency=rebalance_frequency,
    )

    return portfolio_log_returns, summary, cumulative_growth, simple_returns

def plot_strategy_vs_equal_weight(strategy_cum, equal_weight_cum):
    common_index = strategy_cum.index.intersection(equal_weight_cum.index)

    a = strategy_cum.loc[common_index]
    b = equal_weight_cum.loc[common_index]

    plt.figure(figsize=(10, 6))
    plt.plot(a.index, a.values, label="Mean-Variance Portfolio")
    plt.plot(b.index, b.values, label="Equal Weight")
    plt.xlabel("Date")
    plt.ylabel("Growth of $1")
    plt.title("Mean-Variance Portfolio vs Equal Weight")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]

prices, log_returns = download_prices_and_returns(
    tickers=tickers,
    period="5y",
    interval="1d",
    auto_adjust=False,
)

weights_df, portfolio_log_returns, mu_forecasts_df, summary, cumulative_growth, simple_returns = (
    rolling_mean_variance_backtest(
        log_return_df=log_returns,
        window_size=252,
        rebalance_frequency="weekly",
        use_mean_forecast=True,
        use_shrinkage=True,
        risk_aversion=5.0,
    )
)

eq_log_returns, eq_summary, eq_cum, eq_simple = equal_weight_benchmark(
    log_return_df=log_returns,
    rebalance_frequency="weekly",
)

print("Mean-variance summary:")
print(summary)

print("\nEqual-weight summary:")
print(eq_summary)

plot_strategy_vs_equal_weight(cumulative_growth, eq_cum)

Progress: 0.4% (1/261)
Progress: 4.2% (11/261)
Progress: 8.0% (21/261)
Progress: 11.9% (31/261)
Progress: 15.7% (41/261)
Progress: 19.5% (51/261)


KeyboardInterrupt: 

In [ ]:
#===================================
# Wavelet decomposition
#===================================

def build_wavelet_level_return_dfs(
    log_return_df,
    wavelet="db4",
    level=4,
):
    """
    Decompose each asset's return series and build one DataFrame per wavelet level.

    Parameters
    ----------
    log_return_df : pd.DataFrame
        Daily log returns, rows=dates, cols=assets
    wavelet : str
        Wavelet basis
    level : int
        Decomposition level

    Returns
    -------
    level_return_dfs : OrderedDict[str, pd.DataFrame]
        Example keys: A4, D4, D3, D2, D1
    """
    if not isinstance(log_return_df, pd.DataFrame):
        raise TypeError("log_return_df must be a pandas DataFrame.")

    log_return_df = log_return_df.dropna(how="any").copy()
    assets = list(log_return_df.columns)

    # discover level names from first asset
    first_asset = assets[0]
    first_recon = reconstruct_levels(
        log_return_df[first_asset].values,
        wavelet=wavelet,
        level=level,
    )
    level_names = list(first_recon.keys())

    level_return_dfs = OrderedDict()

    for level_name in level_names:
        level_df = pd.DataFrame(index=log_return_df.index)

        for asset in assets:
            recon = reconstruct_levels(
                log_return_df[asset].values,
                wavelet=wavelet,
                level=level,
            )
            level_df[asset] = recon[level_name]

        level_return_dfs[level_name] = level_df.dropna(how="any")

    return level_return_dfs

def forecast_mu_and_cov_wavelet(
    window_returns,
    wavelet="db4",
    level=4,
    selected_levels=None,
    rebalance_frequency="weekly",
    use_mean_forecast=True,
    use_shrinkage=True,
):
    assets = list(window_returns.columns)
    n = len(assets)

    if rebalance_frequency == "daily":
        horizon = 1
    elif rebalance_frequency == "weekly":
        horizon = 5
    elif rebalance_frequency == "monthly":
        horizon = 21
    else:
        raise ValueError("rebalance_frequency must be 'daily', 'weekly', or 'monthly'")

    # historical cross-asset correlation from ORIGINAL returns
    if use_shrinkage:
        base_cov = estimate_shrinkage_covariance(window_returns)
    else:
        base_cov = window_returns.cov()

    base_corr = covariance_to_correlation(base_cov)

    mu_hat = np.zeros(n)
    var_hat = np.zeros(n)
    fit_info = {}

    for i, asset in enumerate(assets):
        y = window_returns[asset].values
        recon = reconstruct_levels(y, wavelet=wavelet, level=level)

        available_levels = list(recon.keys())
        if selected_levels is None:
            chosen = available_levels
        else:
            missing = [lvl for lvl in selected_levels if lvl not in available_levels]
            if missing:
                raise ValueError(f"{asset}: missing levels {missing}")
            chosen = selected_levels

        asset_mean = 0.0
        asset_var = 0.0
        asset_fit = {}

        for lvl in chosen:
            fc = multi_step_arima_garch_forecast(recon[lvl], horizon=horizon)

            if use_mean_forecast:
                asset_mean += fc["mean_forecasts"].sum() if horizon > 1 else fc["mean_forecasts"][0]

            asset_var += fc["var_forecasts"].sum() if horizon > 1 else fc["var_forecasts"][0]
            asset_fit[lvl] = fc

        mu_hat[i] = asset_mean
        var_hat[i] = max(asset_var, 1e-12)
        fit_info[asset] = asset_fit

    D = np.diag(np.sqrt(np.clip(var_hat, 1e-12, None)))
    Sigma_hat = D @ base_corr.values @ D

    mu_hat = pd.Series(mu_hat, index=assets, name="mu_forecast")
    Sigma_hat = pd.DataFrame(Sigma_hat, index=assets, columns=assets)

    return mu_hat, Sigma_hat, fit_info
#===================================
# Backtest
#===================================
def rolling_wavelet_mean_variance_backtest(
    log_return_df,
    wavelet="db4",
    level=4,
    selected_levels=None,
    window_size=252,
    rebalance_frequency="weekly",
    use_mean_forecast=True,
    use_shrinkage=True,
    risk_aversion=1.0,
):
    log_return_df = log_return_df.dropna(how="any").copy()

    if rebalance_frequency == "daily":
        horizon = 1
    elif rebalance_frequency == "weekly":
        horizon = 5
    elif rebalance_frequency == "monthly":
        horizon = 21
    else:
        raise ValueError("rebalance_frequency must be 'daily', 'weekly', or 'monthly'")

    rebalance_dates = compute_rebalance_dates(log_return_df.index, rebalance_frequency)

    weights_list = []
    forecast_list = []
    realized_list = []

    total_steps = len(rebalance_dates)

    for i, date in enumerate(rebalance_dates):
        if i % 10 == 0 or i == total_steps - 1:
            progress = (i + 1) / total_steps * 100
            print(f"Progress: {progress:.1f}% ({i+1}/{total_steps})")

        end_loc = log_return_df.index.get_loc(date)
        if isinstance(end_loc, slice):
            end_loc = end_loc.stop - 1

        if end_loc < window_size - 1:
            continue

        window_returns = log_return_df.iloc[end_loc - window_size + 1 : end_loc + 1]
        if len(window_returns) < window_size:
            continue

        try:
            mu_hat, Sigma_hat, fit_info = forecast_mu_and_cov_wavelet(
                window_returns=window_returns,
                wavelet=wavelet,
                level=level,
                selected_levels=selected_levels,
                rebalance_frequency=rebalance_frequency,
                use_mean_forecast=use_mean_forecast,
                use_shrinkage=use_shrinkage,
            )

            w_t = mean_variance_portfolio_optimization(
                mu_vec=mu_hat,
                cov_matrix=Sigma_hat,
                risk_aversion=risk_aversion,
            )

            # IMPORTANT: realized return comes from ORIGINAL returns
            forward_returns, realized_date = aggregate_forward_returns(
                log_return_df,
                start_loc=end_loc,
                horizon=horizon,
            )

            if forward_returns is None:
                continue

            realized_portfolio_log_return = float(forward_returns.loc[w_t.index] @ w_t.values)

            w_t.name = date
            weights_list.append(w_t)

            mu_row = mu_hat.copy()
            mu_row.name = date
            forecast_list.append(mu_row)

            realized_list.append(
                pd.Series(
                    {
                        "rebalance_date": date,
                        "realized_date": realized_date,
                        "portfolio_log_return": realized_portfolio_log_return,
                    }
                )
            )

        except Exception as e:
            print(f"Skipping {date} due to error: {e}")

    if not weights_list:
        raise ValueError("No valid portfolio weights were computed.")

    weights_df = pd.DataFrame(weights_list)
    weights_df.index.name = "Date"

    mu_forecasts_df = pd.DataFrame(forecast_list)
    mu_forecasts_df.index.name = "Date"

    realized_df = pd.DataFrame(realized_list)
    portfolio_log_returns = pd.Series(
        realized_df["portfolio_log_return"].values,
        index=pd.DatetimeIndex(realized_df["realized_date"]),
        name="portfolio_log_return",
    )

    summary, cumulative_growth, simple_returns = summarize_portfolio_from_log_returns(
        portfolio_log_returns,
        rebalance_frequency=rebalance_frequency,
    )

    return weights_df, portfolio_log_returns, mu_forecasts_df, summary, cumulative_growth, simple_returns


#==================================
# Summary
#==================================
def summarize_selected_levels(level_results):
    """
    Create a summary table across selected levels.
    """
    rows = []
    for lvl, result in level_results.items():
        row = result["summary"].copy()
        row.name = lvl
        rows.append(row)

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows)

def summarize_selected_levels(level_results):
    """
    Create a summary table across selected levels.
    """
    rows = []
    for lvl, result in level_results.items():
        row = result["summary"].copy()
        row.name = lvl
        rows.append(row)

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows)


In [ ]:
def run_level_comparison(
    log_return_df,
    wavelet="db4",
    level=4,
    window_size=252,
    rebalance_frequency="weekly",
    use_mean_forecast=True,
    use_shrinkage=True,
    risk_aversion=1.0,
):
    level_names = [f"A{level}"] + [f"D{i}" for i in range(level, 0, -1)]
    results = OrderedDict()

    for lvl in level_names:
        print(f"\nRunning portfolio using {lvl} forecasts only...")

        weights_df, portfolio_log_returns, mu_forecasts_df, summary, cumulative_growth, simple_returns = (
            rolling_wavelet_mean_variance_backtest(
                log_return_df=log_return_df,
                wavelet=wavelet,
                level=level,
                selected_levels=[lvl],
                window_size=window_size,
                rebalance_frequency=rebalance_frequency,
                use_mean_forecast=use_mean_forecast,
                use_shrinkage=use_shrinkage,
                risk_aversion=risk_aversion,
            )
        )

        results[lvl] = {
            "weights_df": weights_df,
            "portfolio_log_returns": portfolio_log_returns,
            "mu_forecasts_df": mu_forecasts_df,
            "summary": summary,
            "cumulative_growth": cumulative_growth,
            "simple_returns": simple_returns,
        }

    return results

def summarize_level_comparison(results, equal_weight_summary=None):
    rows = []

    for lvl, res in results.items():
        row = res["summary"].copy()
        row.name = lvl
        rows.append(row)

    summary_df = pd.DataFrame(rows)

    if equal_weight_summary is not None:
        ew_row = equal_weight_summary.copy()
        ew_row.name = "EqualWeight"
        summary_df = pd.concat([summary_df, ew_row.to_frame().T])

    return summary_df

def plot_level_comparison(results, equal_weight_cum=None):
    plt.figure(figsize=(12, 7))

    for lvl, res in results.items():
        cum = res["cumulative_growth"]

        if equal_weight_cum is not None:
            common_index = cum.index.intersection(equal_weight_cum.index)
            cum = cum.loc[common_index]

        plt.plot(
            cum.index,
            cum.values,
            label=lvl,
            alpha=0.85
        )

    if equal_weight_cum is not None:
        plt.plot(
            equal_weight_cum.index,
            equal_weight_cum.values,
            label="Equal Weight",
            linewidth=3,
            linestyle="--"
        )

    plt.xlabel("Date")
    plt.ylabel("Growth of $1")
    plt.title("Portfolio Performance by Forecast Level")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_compare_strategies(cum_dict):
    """
    Plot multiple cumulative growth series on the same chart.

    Parameters
    ----------
    cum_dict : dict
        {"label": cumulative_growth_series}
    """
    plt.figure(figsize=(12, 7))

    for label, cum in cum_dict.items():
        plt.plot(cum.index, cum.values, label=label)

    plt.xlabel("Date")
    plt.ylabel("Growth of $1")
    plt.title("Strategy Comparison")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def summarize_strategies(summary_dict):
    rows = []
    for name, summary in summary_dict.items():
        row = summary.copy()
        row.name = name
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
# =========================================
# 1) Download data
# =========================================
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]

prices, log_returns = download_prices_and_returns(
    tickers=tickers,
    period="5y",
    interval="1d",
    auto_adjust=False,
)

# =========================================
# 2) Run equal-weight benchmark on ORIGINAL returns
# =========================================
eq_log_returns, eq_summary, eq_cum, eq_simple = equal_weight_benchmark(
    log_return_df=log_returns,
    rebalance_frequency="weekly",
)

print("Equal-weight summary:")
print(eq_summary)

# =========================================
# 3) Run wavelet level comparison
#    Each portfolio is optimized using ONE level only,
#    but realized returns come from ORIGINAL returns
# =========================================
level_results = run_level_comparison(
    log_return_df=log_returns,
    wavelet="db4",
    level=4,
    window_size=252,
    rebalance_frequency="weekly",
    use_mean_forecast=True,
    use_shrinkage=True,
    risk_aversion=5.0,
)

# =========================================
# 4) Plot all level-based portfolios vs equal weight
# =========================================
plot_level_comparison(level_results, equal_weight_cum=eq_cum)

# =========================================
# 5) Print summary table
# =========================================
summary_df = summarize_level_comparison(
    level_results,
    equal_weight_summary=eq_summary,
)

print("\nLevel comparison summary:")
print(summary_df)

In [ ]:
# A4 only
_, _, _, summary_a4, cum_a4, _ = rolling_wavelet_mean_variance_backtest(
    log_return_df=log_returns,
    wavelet="db4",
    level=4,
    selected_levels=["A4"],
    window_size=252,
    rebalance_frequency="daily",
    use_mean_forecast=True,
    use_shrinkage=True,
    risk_aversion=5.0,
)

# A4 + D2
_, _, _, summary_a4d2, cum_a4d2, _ = rolling_wavelet_mean_variance_backtest(
    log_return_df=log_returns,
    wavelet="db4",
    level=4,
    selected_levels=["A4","D1"],
    window_size=252,
    rebalance_frequency="daily",
    use_mean_forecast=True,
    use_shrinkage=True,
    risk_aversion=5.0,
)

# Equal weight
_, eq_summary, eq_cum, _ = equal_weight_benchmark(
    log_return_df=log_returns,
    rebalance_frequency="daily",
)

In [ ]:
# Plot together
plot_compare_strategies({
    "A4": cum_a4,
    "D2": cum_a4d2,
    "Equal Weight": eq_cum,
})

# Summary table
summary_df = summarize_strategies({
    "A4": summary_a4,
    "D2": summary_a4d2,
    "Equal Weight": eq_summary,
})

print("\nStrategy Summary:")
print(summary_df)